<a href="https://colab.research.google.com/github/Chosencodes/Lung-Segmentation/blob/main/Lung_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install -U gdown
# !gdown https://drive.google.com/file/d/1I1LR7XjyEZ-VBQ-Xruh31V7xExMjlVvi/view?usp=drive_link -O /content/Task06_Lung.tar

In [ ]:
# !tar -xf /content/Task06_Lung.tar -C /content/

In [ ]:
# !du -sh '/content/drive/MyDrive/Task06_Lung'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r '/content/drive/MyDrive/Task06_Lung' '/content/Task06_Lung'

data_path = '/content/Task06_Lung'

In [ ]:
 # !ls -la /content/Task06_Lung

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import os

In [ ]:
root = Path("/content/Task06_Lung/imagesTr")
label = Path("/content/Task06_Lung/labelsTr")

In [ ]:
sample_path = next(root.glob("lung*.nii.gz"))
sample_label_path = label / sample_path.name

In [ ]:
data = nib.load(sample_path)
label = nib.load(sample_label_path)

In [ ]:
ct_scan = data.get_fdata()
mask = label.get_fdata().astype(np.uint8)

In [ ]:
print(data.shape)
print(label.shape)

print(nib.aff2axcodes(data.affine))
print(nib.aff2axcodes(label.affine))
print(data.header.get_zooms())
print(np.unique(mask))

In [ ]:
tumor_pixels = np.sum(mask == 1)
background_pixels = np.sum(mask == 0)

print(tumor_pixels)
print(background_pixels)

print("Tumor %:", tumor_pixels / mask.size * 100)

In [ ]:
# for i, f in enumerate(files):
#     print(i, f)

In [ ]:
case = "lung_054.nii.gz"
print("Patient:", case)

image = nib.load(f"/content/Task06_Lung/imagesTr/{case}").get_fdata()
label = nib.load(f"/content/Task06_Lung/labelsTr/{case}").get_fdata().astype(np.uint8)

tumor_slices = np.where(label.sum(axis=(0, 1)) > 0)[0]

print("Tumor slices:", tumor_slices)

z = tumor_slices[np.argmax(label.sum(axis=(0, 1))[tumor_slices])]

plt.figure(figsize=(7,7))

plt.imshow(np.clip(image[:, :, z], -1000, 400).T,
           cmap="gray",
           origin="lower")

plt.imshow(np.ma.masked_where(label[:, :, z] == 0,
                              label[:, :, z]).T,
           cmap="autumn",
           alpha=0.6,
           origin="lower")

plt.title(f"{case} - Slice {z}")
plt.axis("off")
plt.show()

# **Preprocessing**

In [ ]:
!pip install torchio nibabel scikit-learn tqdm -q

In [ ]:
import torchio as tio
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

In [ ]:
root = Path("/content/Task06_Lung/imagesTr")
label = Path("/content/Task06_Lung/labelsTr")

In [ ]:
image_path = sorted(root.glob("lung*.nii.gz"))
label_path = sorted(label.glob("lung*.nii.gz"))
print("images:", len(image_path), "labels:", len(label_path))

In [ ]:
def make_subject(image_path,label_path):
  subjects = []
  for img_path,lbl_path in zip(image_path,label_path):
    subject = tio.Subject(
        mri = tio.ScalarImage(img_path),
        mask = tio.LabelMap(lbl_path)
    )
    subjects.append(subject)
  return subjects

all_subject = make_subject(image_path,label_path)

In [ ]:
train_subject,val_subject = train_test_split(
    all_subject,
    test_size = 0.20,
    random_state=42
)

In [ ]:
print("Total:", len(all_subject))
print("Train:", len(train_subject), "| Val:", len(val_subject))

In [ ]:
base_transform = tio.Compose([
    tio.ToCanonical(),
    tio.Resample(1.5),
    tio.Clamp(-1000,400),
    tio.RescaleIntensity(
        out_min_max=(0, 1),
        in_min_max=(-1000, 400),
    ),
])

In [ ]:
train_transform = tio.Compose([
    base_transform,
    tio.RandomFlip(axes=(0,), flip_probability=0.5),
    tio.RandomAffine(scales=(0.9, 1.1), degrees=10),
    tio.RandomNoise(std=(0, 0.03)),
])

val_transform = tio.Compose([
    base_transform,
])

In [ ]:
train_dataset = tio.SubjectsDataset(train_subject, transform=train_transform)
val_dataset = tio.SubjectsDataset(val_subject, transform=val_transform)

In [ ]:
patch_size = (96, 96, 96)

sampler = tio.LabelSampler(
    patch_size=patch_size,
    label_name="mask",
    label_probabilities={0: 0.2, 1: 0.8},
)

train_queue = tio.Queue(
    subjects_dataset=train_dataset,
    max_length=32,
    samples_per_volume=4,
    sampler=sampler,
    num_workers=0,
    shuffle_subjects=True,
    shuffle_patches=True,
)

val_queue = tio.Queue(
    subjects_dataset=val_dataset,
    max_length=32,
    samples_per_volume=4,
    sampler=sampler,
    num_workers=0,
    shuffle_subjects=False,
    shuffle_patches=False,
)

In [ ]:

train_loader = tio.SubjectsLoader(train_queue, batch_size=2)
batch = next(iter(train_loader))
print("CT patch:  ", batch['mri'][tio.DATA].shape)
print("Mask patch:", batch['mask'][tio.DATA].shape)
print("Mask values:", torch.unique(batch['mask'][tio.DATA]))

# **Model**

In [ ]:
!pip install monai pytorch-lightning -q

In [ ]:
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
import pytorch_lightning as pl
from torch.utils.data import DataLoader, Dataset
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

In [ ]:
train_loader = tio.SubjectsLoader(train_queue,batch_size=2,num_workers=0)
val_loader = tio.SubjectsLoader(val_queue,batch_size=2,num_workers=0)

In [ ]:
def build_model():
  return UNet(
      spatial_dims = 3,
      in_channels = 1,
      out_channels = 1,
      channels=(32,64,128,256,512),
      strides=(2,2,2,2),
      num_res_units=2,
      dropout=0.1,
  )

In [ ]:
class LungTumorSegmentation(pl.LightningModule):
  def __init__(self,lr=1e-4):
    super().__init__()

    self.save_hyperparameters()
    self.model = build_model()
    self.loss_fn = DiceCELoss(sigmoid=True,lambda_dice=0.7,lambda_ce=0.3)
    self.metrics = DiceMetric(include_background=False,reduction="mean")

  def forward(self,x):
    return self.model(x)

  def training_step(self,batch,batch_idx):
    mri = batch["mri"]["data"]
    mask = batch["mask"]["data"].float()
    batch_size = mri.shape[0]
    pred = self.model(mri)
    loss = self.loss_fn(pred,mask)

    self.log("Train loss", loss, prog_bar=True,batch_size=batch_size)

    if batch_idx % 10 == 0:
      self.log_images(mri.cpu(),pred.cpu(),mask.cpu(),"train")

    return loss

  def validation_step(self,batch,batch_idx):
    mri = batch["mri"]["data"]
    mask = batch["mask"]["data"].float()
    batch_size = mri.shape[0]
    pred = self.model(mri)
    loss = self.loss_fn(pred,mask)

    pred_binary = (torch.sigmoid(pred)>0.5).float()
    self.metrics(pred_binary,mask)

    self.log("Val loss", loss,prog_bar=True,batch_size=batch_size)

    if batch_idx % 5 == 0:
      self.log_images(mri.cpu(),pred.cpu(),mask.cpu(),"val")
    return loss

  def on_validation_epoch_end(self):
    dice_score = self.metrics.aggregate().item()
    self.metrics.reset()
    self.log("Dice score",dice_score,prog_bar=True)

  def log_images(self, mri, pred, mask, name):
    d = mri.shape[-1] // 2
    pred_binary = (torch.sigmoid(pred) > 0.5).float()


    ct = mri[0, 0, :, :, d]

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(ct, cmap="gray")
    gt = np.ma.masked_where(mask[0, 0, :, :, d] == 0, mask[0, 0, :, :, d])
    axes[0].imshow(gt, alpha=0.6, cmap="autumn")
    axes[0].set_title("Ground Truth")
    axes[0].axis("off")

    axes[1].imshow(ct, cmap="gray")
    pr = np.ma.masked_where(pred_binary[0, 0, :, :, d] == 0, pred_binary[0, 0, :, :, d])
    axes[1].imshow(pr, alpha=0.6, cmap="autumn")
    axes[1].set_title("Prediction")
    axes[1].axis("off")

    plt.suptitle(f"{name} — depth slice {d}", fontsize=12)
    plt.tight_layout()
    self.logger.experiment.add_figure(name, fig, self.global_step)
    plt.close(fig)

  def configure_optimizers(self):
    optimizer = torch.optim.Adam(self.model.parameters(),lr=self.hparams.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="min",factor=0.5,patience=5,)
    return {
    "optimizer": optimizer,
    "lr_scheduler": {
        "scheduler": scheduler,
        "monitor": "Val loss",
    },
}


In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor= "Val loss",
    mode = "min",
    save_top_k= 10,
    filename= "best_checkpoiint"
)

early_stop_callback = EarlyStopping(
    monitor= "Val loss",
    patience= 5,
    mode= "min",
)

In [ ]:
torch.manual_seed(42)
model = LungTumorSegmentation()

In [ ]:
trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs = 150,
    precision="16-mixed",
    callbacks=[checkpoint_callback, early_stop_callback],
    logger=TensorBoardLogger(save_dir="./logs", name="lung"),
    log_every_n_steps= 1,
)

In [ ]:
trainer.fit(model, train_loader, val_loader)

In [ ]:
import time

start = time.time()

batch = next(iter(train_loader))

print("Loaded in", time.time() - start, "seconds")
print(batch["mri"]["data"].shape)